# AEMO DUID Network Constraint Forecast Demo

This notebook is the small, readable demo version for the Proa Analytics internship application.

Goal: for a selected `DUID`, map it to relevant AEMO generic constraints, build historical binding labels from public dispatch data, add weather/time features, train a baseline ML model, and save a bundle that can be queried by the FastAPI service.

## 0. Setup

Run from the repository root after installing `requirements.txt`.

The default example uses `OAKEY1SF` with approximate Oakey, QLD coordinates. Replace the DUID and coordinates for another generator.

In [ ]:
from pathlib import Path
import sys

# Make package importable when the notebook is opened from notebooks/
REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

import joblib
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, classification_report, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from src.aemo_constraint_demo.aemo_client import load_next_day_dispatch_tables, read_monthly_table
from src.aemo_constraint_demo.features import (
    add_time_features,
    build_label_frame,
    numericise_feature_columns,
    relevant_constraints_for_duid,
    select_feature_columns,
)
from src.aemo_constraint_demo.weather import fetch_open_meteo_history, merge_weather_to_intervals

In [ ]:
DUID = "OAKEY1SF"
LATITUDE = -27.43
LONGITUDE = 151.72
DAYS = 14          # keep small for demo; increase to 21-45 for better backtesting
CACHE_DIR = REPO_ROOT / "data" / "cache"
MODEL_OUT = REPO_ROOT / "artifacts" / f"{DUID.lower()}_demo.joblib"
BIND_THRESHOLD = 0.0
INCLUDE_DISPATCH_FEATURES = False  # False keeps query-time features simple: time + weather

## 1. Download standing-data tables

These tables map a DUID to the generic constraints that can affect it.

- `DUDETAIL`: DUID metadata and connection point.
- `SPDCONNECTIONPOINTCONSTRAINT`: LHS factors linking connection points to generic constraints.
- `GENCONDATA`, `GENCONSET`, and `GENCONSETINVOKE`: used later for the planned/invoked constraints in the API response.

In [ ]:
dudetail = read_monthly_table("DUDETAIL", cache_dir=CACHE_DIR)
spd_cp = read_monthly_table("SPDCONNECTIONPOINTCONSTRAINT", cache_dir=CACHE_DIR)
gencondata = read_monthly_table("GENCONDATA", cache_dir=CACHE_DIR)
genconset = read_monthly_table("GENCONSET", cache_dir=CACHE_DIR)
genconsetinvoke = read_monthly_table("GENCONSETINVOKE", cache_dir=CACHE_DIR)

print(dudetail.shape, spd_cp.shape, gencondata.shape, genconset.shape, genconsetinvoke.shape)

## 2. Map DUID → connection point → GENCONID

This is the core domain step. The model is not forecasting a generic global label; it is forecasting whether constraints relevant to this DUID are likely to bind.

In [ ]:
relevant_genconids, connection_point, lhs_rows = relevant_constraints_for_duid(DUID, dudetail, spd_cp)
print("DUID:", DUID)
print("Connection point:", connection_point)
print("Relevant GENCONIDs:", len(relevant_genconids))
lhs_rows.head()

## 3. Download recent Next Day Dispatch files

For a small demo, use daily Next Day Dispatch files instead of large monthly dispatch archives. These include unit dispatch and constraint solutions for each 5-minute interval of the previous trading day.

In [ ]:
dispatch = load_next_day_dispatch_tables(
    days=DAYS,
    table_names=["DISPATCHLOAD", "DISPATCHCONSTRAINT"],
    cache_dir=CACHE_DIR,
)

print("DISPATCHLOAD", dispatch["DISPATCHLOAD"].shape)
print("DISPATCHCONSTRAINT", dispatch["DISPATCHCONSTRAINT"].shape)

## 4. Build labels

Target definition:

```text
1 if any relevant GENCONID has MARGINALVALUE > 0 or VIOLATIONDEGREE > 0
0 otherwise
```

The interval grid comes from `DISPATCHLOAD` rows for the selected DUID.

In [ ]:
frame = build_label_frame(
    dispatchload=dispatch["DISPATCHLOAD"],
    dispatchconstraint=dispatch["DISPATCHCONSTRAINT"],
    duid=DUID,
    relevant_genconids=relevant_genconids,
    bind_threshold=BIND_THRESHOLD,
)

frame[["SETTLEMENTDATE", "DUID", "target_constrained", "binding_constraint_ids"]].head()

In [ ]:
print("Rows:", len(frame))
print("Positive rate:", frame["target_constrained"].mean())
frame["target_constrained"].value_counts(dropna=False)

## 5. Add time and weather features

For a minimal queryable model, default features are time-of-day/day-of-week/month plus weather. A production system should also add AEMO forecast demand, interconnector flows, outage schedules, P5MIN/Predispatch constraints, and DUID-level forecast generation where available.

In [ ]:
frame = add_time_features(frame)

start_date = frame["SETTLEMENTDATE"].min().date()
end_date = frame["SETTLEMENTDATE"].max().date()
weather = fetch_open_meteo_history(LATITUDE, LONGITUDE, start_date, end_date)
frame = merge_weather_to_intervals(frame, weather)

feature_cols = select_feature_columns(frame, include_dispatch_features=INCLUDE_DISPATCH_FEATURES)
frame = numericise_feature_columns(frame, feature_cols)

feature_cols

## 6. Chronological train/test split

This is a time-series problem, so avoid random shuffling. The last 20% of intervals are held out as a simple forward test set.

In [ ]:
frame = frame.sort_values("SETTLEMENTDATE").reset_index(drop=True)
split = int(len(frame) * 0.8)
train_df = frame.iloc[:split].copy()
test_df = frame.iloc[split:].copy()

X_train, y_train = train_df[feature_cols], train_df["target_constrained"].astype(int)
X_test, y_test = test_df[feature_cols], test_df["target_constrained"].astype(int)

print(train_df["SETTLEMENTDATE"].min(), "to", train_df["SETTLEMENTDATE"].max(), len(train_df))
print(test_df["SETTLEMENTDATE"].min(), "to", test_df["SETTLEMENTDATE"].max(), len(test_df))

## 7. Train baseline model

Balanced logistic regression is intentionally simple. The purpose is to show a clean end-to-end forecasting workflow, not to claim production-grade accuracy.

In [ ]:
model = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42)),
    ]
)
model.fit(X_train, y_train)

score = model.predict_proba(X_test)[:, 1]
pred = (score >= 0.5).astype(int)

metrics = {
    "rows": int(len(frame)),
    "positive_rate": float(frame["target_constrained"].mean()),
    "test_average_precision": float(average_precision_score(y_test, score)) if y_test.nunique() > 1 else None,
    "test_roc_auc": float(roc_auc_score(y_test, score)) if y_test.nunique() > 1 else None,
}
metrics

In [ ]:
print(classification_report(y_test, pred, zero_division=0))

## 8. Save model bundle

The API loads this bundle and returns a JSON forecast for the trained DUID.

In [ ]:
MODEL_OUT.parent.mkdir(parents=True, exist_ok=True)

# Keep only rows needed for planned/invoked constraints.
gencondata_u = gencondata.copy()
gencondata_u.columns = [str(c).strip().upper() for c in gencondata_u.columns]
genconset_u = genconset.copy()
genconset_u.columns = [str(c).strip().upper() for c in genconset_u.columns]
genconsetinvoke_u = genconsetinvoke.copy()
genconsetinvoke_u.columns = [str(c).strip().upper() for c in genconsetinvoke_u.columns]

relevant_gdata = gencondata_u[gencondata_u["GENCONID"].astype(str).str.upper().isin(relevant_genconids)]
relevant_gset = genconset_u[genconset_u["GENCONID"].astype(str).str.upper().isin(relevant_genconids)]
relevant_sets = set(relevant_gset.get("GENCONSETID", pd.Series(dtype=str)).astype(str).str.upper())
if relevant_sets and "GENCONSETID" in genconsetinvoke_u.columns:
    relevant_invocations = genconsetinvoke_u[
        genconsetinvoke_u["GENCONSETID"].astype(str).str.upper().isin(relevant_sets)
    ]
else:
    relevant_invocations = pd.DataFrame()

bundle = {
    "model": model,
    "feature_columns": feature_cols,
    "duid": DUID.upper(),
    "connection_point": connection_point,
    "latitude": LATITUDE,
    "longitude": LONGITUDE,
    "relevant_genconids": relevant_genconids,
    "lhs_rows": lhs_rows,
    "planned_gencondata": relevant_gdata,
    "planned_genconset": relevant_gset,
    "planned_genconsetinvoke": relevant_invocations,
    "metrics": metrics,
    "bind_threshold": BIND_THRESHOLD,
    "include_dispatch_features": INCLUDE_DISPATCH_FEATURES,
    "no_weather": False,
}
joblib.dump(bundle, MODEL_OUT)
MODEL_OUT

## 9. Query example

After saving the model, start the API from the repository root:

```bash
uvicorn app:app --reload
```

Then query:

```bash
curl "http://127.0.0.1:8000/forecast/OAKEY1SF?horizon_hours=24&model_path=artifacts/oakey1sf_demo.joblib"
```

In [ ]:
from src.aemo_constraint_demo.forecast import forecast_from_bundle

result = forecast_from_bundle(MODEL_OUT, duid=DUID, horizon_hours=2)
pd.DataFrame(result["forecast"]).head(12)

## Interpretation notes

- High probability means the model sees a time/weather pattern similar to previously constrained intervals for this DUID.
- This does not prove physical causality. Weather/time features are a small demo signal; production requires system topology, outage schedules, interconnector flow forecasts, demand forecasts, and P5MIN/Predispatch constraint solutions.
- The most important achievement here is the end-to-end pipeline: public AEMO data → DUID-relevant labels → model → query service.